# State-based History-aware Artificial Reinforcement Interpretable Kernel" (Sharik)
## Open AI Gym - Atari - Breakout - Experiential Learning - Exploring Downsampling and Bordering

- TODO


In [1]:
import os, sys
import pickle
import pandas as pd
import queue
import matplotlib.pyplot as plt
import random
import math
import numpy as np

## Experiment with Bordering and Downsampling

In [10]:
#top/border/bottom=93/8/0 ds=1
#grep avg sim_test5/*_bot0*
#sim_test5/exp1080000_s2ds1ss10lm2sc2cs2tu0er0mc0_bot0.txt:score_avg=8.0; steps_tot=1080000; steps_avg=2693.3; lives_avg=0.2; lapse_avg="0:00:05.548562"; time="0:08:06.344161"
#sim_test5/exp1080000_s3ds1ss10lm2sc2cs2tu0er0mc0_bot0.txt:score_avg=7.2; steps_tot=1080000; steps_avg=1767.6; lives_avg=0.0; lapse_avg="0:00:01.735763"; time="0:07:02.768250"
#sim_test5/exp1080000_s41ds1ss10lm2sc2cs2tu0er0mc0_bot0.txt:score_avg=11.3; steps_tot=1080000; steps_avg=2049.3; lives_avg=0.0; lapse_avg="0:00:00.528796"; time="0:07:21.858110"

#top/border/bottom=0/0/0 ds=1
#grep avg sim_test5/*noborder.txt      
#sim_test5/exp1080000_s2ds1ss10lm2sc2cs2tu0er0mc0_noborder.txt:score_avg=1.1; steps_tot=1080000; steps_avg=17142.9; lives_avg=3.7; lapse_avg="0:00:04.435827"; time="0:05:08.372708"
#sim_test5/exp1080000_s3ds1ss10lm2sc2cs2tu0er0mc0_noborder.txt:score_avg=1.1; steps_tot=1080000; steps_avg=18000.0; lives_avg=3.8; lapse_avg="0:00:05.166166"; time="0:05:12.741660"
#sim_test5/exp1080000_s41ds1ss10lm2sc2cs2tu0er0mc0_noborder.txt:score_avg=1.2; steps_tot=1080000; steps_avg=9818.2; lives_avg=1.5; lapse_avg="0:00:03.701141"; time="0:05:04.182612"
#TODO see how to imporve it with a better "machine vision" - fix racket_ball_x to use diff!!! 

#top/border/bottom=93/8/17 ds=1
#grep avg sim_test5/*bot17.txt                                                                                           
#sim_test5/exp1080000_s2ds1ss10lm2sc2cs2tu0er0mc0_bot17.txt:score_avg=8.0; steps_tot=1080000; steps_avg=2693.3; lives_avg=0.2; lapse_avg="0:00:05.655163"; time="0:07:12.212485"
#sim_test5/exp1080000_s3ds1ss10lm2sc2cs2tu0er0mc0_bot17.txt:score_avg=7.2; steps_tot=1080000; steps_avg=1767.6; lives_avg=0.0; lapse_avg="0:00:01.384369"; time="0:06:38.386910"
#sim_test5/exp1080000_s41ds1ss10lm2sc2cs2tu0er0mc0_bot17.txt:score_avg=11.3; steps_tot=1080000; steps_avg=2049.3; lives_avg=0.0; lapse_avg="0:00:00.711849"; time="0:06:34.625463"

#top/border/bottom=93/8/17 ds=2
#!!!

#top/border/bottom=93/8/17 ds=3
#!!!

#top/border/bottom=93/8/17 ds=4
#!!!

#top/border/bottom=93/8/17 ds=5
#!!!




In [69]:
def downsample_max(arr, factor, dtype):
    if type(arr) == list:
        arr = np.array(arr)
    h, w = arr.shape
    # Crop to multiples of factor
    new_h, new_w = h // factor, w // factor
    cropped = arr[:new_h * factor, :new_w * factor]
    # Reshape to group into blocks and take the max
    reshaped = cropped.reshape(new_h, factor, new_w, factor)
    return reshaped.max(axis=(1, 3))

def downsample_mean(arr, factor, dtype):
    if type(arr) == list:
        arr = np.array(arr)
    h, w = arr.shape
    # Crop to multiples of factor
    new_h, new_w = h // factor, w // factor
    cropped = arr[:new_h * factor, :new_w * factor]
    # Reshape to group into blocks and take the max
    reshaped = cropped.reshape(new_h, factor, new_w, factor)
    return reshaped.mean(axis=(1, 3)).astype(int)

arr = np.random.randint(0, 10, (6, 8))
print(arr)
downsampled = downsample_max(arr, 2, type(arr[0][0]))
print(downsampled)  # Outputs integers, shape (3, 4)


[[2 4 3 0 7 6 3 7]
 [6 7 5 9 6 1 1 3]
 [5 6 3 9 4 1 2 7]
 [3 4 2 4 3 0 7 1]
 [4 5 5 3 4 5 5 7]
 [7 0 1 3 5 6 2 5]]
[[7 9 7 7]
 [6 9 4 7]
 [7 5 6 7]]


In [65]:
arr = np.array([[1, 1, 2, 2],[1, 1, 2, 2],[3, 3, 4, 4],[3, 3, 4, 4]])
print(arr)
print(downsample_max(arr, 2, type(arr[0][0])))
print(downsample_mean(arr, 2, type(arr[0][0])))


[[1 1 2 2]
 [1 1 2 2]
 [3 3 4 4]
 [3 3 4 4]]
[[1 2]
 [3 4]]
[[1 2]
 [3 4]]


In [66]:
arr = np.array([[0, 1, 1, 2],[1, 2, 2, 3],[2, 3, 3, 4],[3, 4, 4, 5]])
print(arr)
print(downsample_max(arr, 2, type(arr[0][0])))
print(downsample_mean(arr, 2, type(arr[0][0])))


[[0 1 1 2]
 [1 2 2 3]
 [2 3 3 4]
 [3 4 4 5]]
[[2 3]
 [4 5]]
[[1 2]
 [3 4]]


In [68]:
type([[0, 1, 1, 2],[1, 2, 2, 3],[2, 3, 3, 4],[3, 4, 4, 5]])

list